# **Ingeniería de características**
---

## **1. Librerías**
---

In [1]:
# Importar librerías
import sys
from pathlib import Path

# Añadir el directorio raíz del proyecto al path de Python PRIMERO
proyecto_root = str(Path('.').resolve().parent)
if proyecto_root not in sys.path:
    sys.path.insert(0, proyecto_root)

import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [2]:
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.float_format", "{:,.4f}".format)

## **2. Análisis de la base de datos**
---

### **2.1 Cargue de la base de datos**
---

In [3]:
# Conectar con directorio raíz
PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))
PROJECT_ROOT

WindowsPath('c:/Users/cecor/OneDrive - Universidad Nacional de Colombia/GitHub/MSc-Thesis-Financial-Fraud-Detection-Models')

In [6]:
DATA_PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
INPUT_PATH = DATA_PROCESSED_DIR / "card_transactions_clean.parquet"
OUTPUT_TRANSACTION_FEATURES_PATH = DATA_PROCESSED_DIR / "transactions_modeling.parquet"
OUTPUT_USER_CARD_SNAPSHOT_PATH = DATA_PROCESSED_DIR / "user_card_snapshot_features.parquet"

INPUT_PATH.exists()

True

In [19]:
df = pd.read_parquet(INPUT_PATH)
df.shape

(24386900, 23)

### **2.2 Validación inicial de la base a usar**
---

In [20]:
required_columns = [    "user",
    "card",
    "year",
    "month",
    "day",
    "time",
    "amount",
    "amount_abs",
    "amount_is_negative",
    "use_chip",
    "merchant_name",
    "merchant_city",
    "merchant_state",
    "zip",
    "mcc",
    "is_fraud",
    "merchant_state_was_missing",
    "zip_was_missing",
    "user_card_id",
    "hour",
    "datetime",
    "day_of_week",
    "is_weekend",]

missing_columns = [col for col in required_columns if col not in df.columns]
if len(missing_columns) == 0:
    print("No hay columnas faltantes")
else:
    print(f"Las columnas faltantes son: {', '.join(missing_columns)}")

No hay columnas faltantes


### **2.3 Variables transaccionales simples**
---

In [21]:
# Crear nueva característica: logaritmo del monto absoluto
df["amount_log"] = np.log1p(df["amount_abs"])

In [22]:
# Crear características binarias para el tipo de transacción
df["is_online_transaction"] = (
    df["use_chip"].astype(str).str.lower().str.contains("online")
).astype(int)

df["is_chip_transaction"] = (
    df["use_chip"].astype(str).str.lower().str.contains("chip")
).astype(int)

df["is_swipe_transaction"] = (
    df["use_chip"].astype(str).str.lower().str.contains("swipe")
).astype(int)

### **2.3 Variables temporales**
---

In [23]:
df["datetime"] = pd.to_datetime(df["datetime"])

In [24]:
df["year_month"] = df["datetime"].dt.to_period("M").dt.to_timestamp()
df["quarter"] = df["datetime"].dt.quarter
df["day_of_month"] = df["datetime"].dt.day

In [25]:
df[["datetime", "year_month", "year", "month", "quarter", "day_of_month", "hour", "day_of_week", "is_weekend"]].head()

,datetime,year_month,year,month,quarter,day_of_month,hour,day_of_week,is_weekend
0,2002-09-01 06:21:00,2002-09-01,2002,9,3,1,6,6,1
1,2002-09-01 06:42:00,2002-09-01,2002,9,3,1,6,6,1
2,2002-09-02 06:22:00,2002-09-01,2002,9,3,2,6,0,0
3,2002-09-02 17:45:00,2002-09-01,2002,9,3,2,17,0,0
4,2002-09-03 06:23:00,2002-09-01,2002,9,3,3,6,1,0


### **2.3 Definir el momento T0**
---

In [29]:
evaluation_month = df["year_month"].max()
evaluation_month

Timestamp('2020-02-01 00:00:00')

El momento T0 de la base, es decir, el punto de referencia a partir del cual se crean las variables transaccionales es feb-2020.

In [31]:
window_3m_start = evaluation_month - pd.DateOffset(months=3)
window_6m_start = evaluation_month - pd.DateOffset(months=6)
window_12m_start = evaluation_month - pd.DateOffset(months=12)

window_3m_start, window_6m_start, window_12m_start

(Timestamp('2019-11-01 00:00:00'),
 Timestamp('2019-08-01 00:00:00'),
 Timestamp('2019-02-01 00:00:00'))

In [32]:
df = df.sort_values(["user_card_id", "datetime"]).reset_index(drop=True)

In [34]:
df.head(2)

,user,card,year,month,day,time,amount,use_chip,merchant_name,merchant_city,merchant_state,zip,mcc,is_fraud,merchant_state_was_missing,zip_was_missing,user_card_id,amount_is_negative,amount_abs,hour,datetime,day_of_week,is_weekend,amount_log,is_online_transaction,is_chip_transaction,is_swipe_transaction,year_month,quarter,day_of_month
0,0,0,2002,9,1,06:21,134.0900,Swipe Transaction,3527213246127876953,La Verne,CA,91750,5300,0,0,0,0_0,0,134.0900,6,2002-09-01 06:21:00,6,1,4.9059,0,0,1,2002-09-01,3,1
1,0,0,2002,9,1,06:42,38.4800,Swipe Transaction,-727612092139916043,Monterey Park,CA,91754,5411,0,0,0,0_0,0,38.4800,6,2002-09-01 06:42:00,6,1,3.6758,0,0,1,2002-09-01,3,1


### **2.3 Variables historicas**
---

In [36]:
#Suma histórica de montos y transacciones
df["uc_tx_count_hist"] = df.groupby("user_card_id").cumcount()

df["uc_amount_sum_hist"] = (df.groupby("user_card_id")["amount_abs"].cumsum() - df["amount_abs"])

In [37]:
#Promedio histórico de montos por user_card_id
df["uc_amount_mean_hist"] = (
    df["uc_amount_sum_hist"] / df["uc_tx_count_hist"].replace(0, np.nan))

df["uc_amount_mean_hist"] = df["uc_amount_mean_hist"].fillna(0)

In [38]:
#maximo histórico de monto por user_card_id
df["uc_amount_max_hist"] = (
    df.groupby("user_card_id")["amount_abs"]
    .cummax()
    .groupby(df["user_card_id"])
    .shift(1)
    .fillna(0))

In [39]:
#tiempo desde la última transacción
df["uc_prev_datetime"] = df.groupby("user_card_id")["datetime"].shift(1)

df["uc_days_since_prev_tx"] = (
    (df["datetime"] - df["uc_prev_datetime"])
    .dt.total_seconds()
    .div(86400))

df["uc_days_since_prev_tx"] = df["uc_days_since_prev_tx"].fillna(-1)

In [40]:
df.head()

,user,card,year,month,day,time,amount,use_chip,merchant_name,merchant_city,merchant_state,zip,mcc,is_fraud,merchant_state_was_missing,zip_was_missing,user_card_id,amount_is_negative,amount_abs,hour,datetime,day_of_week,is_weekend,amount_log,is_online_transaction,is_chip_transaction,is_swipe_transaction,year_month,quarter,day_of_month,uc_tx_count_hist,uc_amount_sum_hist,uc_amount_mean_hist,uc_amount_max_hist,uc_prev_datetime,uc_days_since_prev_tx
0,0,0,2002,9,1,06:21,134.0900,Swipe Transaction,3527213246127876953,La Verne,CA,91750,5300,0,0,0,0_0,0,134.0900,6,2002-09-01 06:21:00,6,1,4.9059,0,0,1,2002-09-01,3,1,0,0.0000,0.0000,0.0000,NaT,-1.0000
1,0,0,2002,9,1,06:42,38.4800,Swipe Transaction,-727612092139916043,Monterey Park,CA,91754,5411,0,0,0,0_0,0,38.4800,6,2002-09-01 06:42:00,6,1,3.6758,0,0,1,2002-09-01,3,1,1,134.0900,134.0900,134.0900,2002-09-01 06:21:00,0.0146
2,0,0,2002,9,2,06:22,120.3400,Swipe Transaction,-727612092139916043,Monterey Park,CA,91754,5411,0,0,0,0_0,0,120.3400,6,2002-09-02 06:22:00,0,0,4.7986,0,0,1,2002-09-01,3,2,2,172.5700,86.2850,134.0900,2002-09-01 06:42:00,0.9861
3,0,0,2002,9,2,17:45,128.9500,Swipe Transaction,3414527459579106770,Monterey Park,CA,91754,5651,0,0,0,0_0,0,128.9500,17,2002-09-02 17:45:00,0,0,4.8671,0,0,1,2002-09-01,3,2,3,292.9100,97.6367,134.0900,2002-09-02 06:22:00,0.4743
4,0,0,2002,9,3,06:23,104.7100,Swipe Transaction,5817218446178736267,La Verne,CA,91750,5912,0,0,0,0_0,0,104.7100,6,2002-09-03 06:23:00,1,0,4.6607,0,0,1,2002-09-01,3,3,4,421.8600,105.4650,134.0900,2002-09-02 17:45:00,0.5264


### **2.3 Variables mensuales por user-id**
---

Además de variables acumuladas, se construyen agregados por ventanas de tiempo. Para simplificar y evitar fuga de información, se usan meses completos anteriores al mes de cada transacción.

Por ejemplo, para una transacción en junio de 2018, la ventana de 3 meses usa marzo, abril y mayo de 2018, no incluye junio.

In [41]:
monthly_user_card = (
    df
    .groupby(["user_card_id", "year_month"])
    .agg(
        uc_month_tx_count=("is_fraud", "size"),
        uc_month_amount_sum=("amount_abs", "sum"),
        uc_month_amount_mean=("amount_abs", "mean"),
        uc_month_amount_max=("amount_abs", "max"),
    )
    .reset_index()
    .sort_values(["user_card_id", "year_month"])
)

monthly_user_card.head()

,user_card_id,year_month,uc_month_tx_count,uc_month_amount_sum,uc_month_amount_mean,uc_month_amount_max
0,0_0,2002-09-01,89,"7,691.4400",86.4207,199.7100
1,0_0,2002-10-01,78,"8,183.0900",104.9114,"1,049.8200"
2,0_0,2002-11-01,78,"7,833.4400",100.4287,379.7300
3,0_0,2002-12-01,84,"7,669.4900",91.3035,255.0000
4,0_0,2003-01-01,50,"4,341.2300",86.8246,432.2200


In [42]:
monthly_user_card["uc_tx_count_3m"] = (
    monthly_user_card
    .groupby("user_card_id")["uc_month_tx_count"]
    .transform(lambda s: s.shift(1).rolling(window=3, min_periods=1).sum())
)

monthly_user_card["uc_tx_count_6m"] = (
    monthly_user_card
    .groupby("user_card_id")["uc_month_tx_count"]
    .transform(lambda s: s.shift(1).rolling(window=6, min_periods=1).sum())
)

monthly_user_card["uc_amount_sum_3m"] = (
    monthly_user_card
    .groupby("user_card_id")["uc_month_amount_sum"]
    .transform(lambda s: s.shift(1).rolling(window=3, min_periods=1).sum())
)

monthly_user_card["uc_amount_sum_6m"] = (
    monthly_user_card
    .groupby("user_card_id")["uc_month_amount_sum"]
    .transform(lambda s: s.shift(1).rolling(window=6, min_periods=1).sum())
)

In [44]:
# promedios por ventana
monthly_user_card["uc_amount_mean_3m"] = (
    monthly_user_card["uc_amount_sum_3m"] / monthly_user_card["uc_tx_count_3m"]
)

monthly_user_card["uc_amount_mean_6m"] = (
    monthly_user_card["uc_amount_sum_6m"] / monthly_user_card["uc_tx_count_6m"]
)


In [45]:
window_columns = [
    "uc_tx_count_3m",
    "uc_tx_count_6m",
    "uc_amount_sum_3m",
    "uc_amount_sum_6m",
    "uc_amount_mean_3m",
    "uc_amount_mean_6m",
]

monthly_user_card[window_columns] = monthly_user_card[window_columns].fillna(0)

In [46]:
monthly_user_card_features = monthly_user_card[
    [
        "user_card_id",
        "year_month",
        "uc_tx_count_3m",
        "uc_tx_count_6m",
        "uc_amount_sum_3m",
        "uc_amount_sum_6m",
        "uc_amount_mean_3m",
        "uc_amount_mean_6m",
    ]
]

monthly_user_card_features.head()

,user_card_id,year_month,uc_tx_count_3m,uc_tx_count_6m,uc_amount_sum_3m,uc_amount_sum_6m,uc_amount_mean_3m,uc_amount_mean_6m
0,0_0,2002-09-01,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000
1,0_0,2002-10-01,89.0000,89.0000,"7,691.4400","7,691.4400",86.4207,86.4207
2,0_0,2002-11-01,167.0000,167.0000,"15,874.5300","15,874.5300",95.0571,95.0571
3,0_0,2002-12-01,245.0000,245.0000,"23,707.9700","23,707.9700",96.7672,96.7672
4,0_0,2003-01-01,240.0000,329.0000,"23,686.0200","31,377.4600",98.6917,95.3722


### **2.3 Union al conjunto de datos transaccional**
---

In [47]:
df = df.merge(
    monthly_user_card_features,
    on=["user_card_id", "year_month"],
    how="left",
)

In [48]:
df[window_columns] = df[window_columns].fillna(0)

In [49]:
df[
    [
        "user_card_id",
        "datetime",
        "year_month",
        "amount_abs",
        "uc_tx_count_3m",
        "uc_tx_count_6m",
        "uc_amount_sum_3m",
        "uc_amount_sum_6m",
        "uc_amount_mean_3m",
        "uc_amount_mean_6m",
    ]
].head()

,user_card_id,datetime,year_month,amount_abs,uc_tx_count_3m,uc_tx_count_6m,uc_amount_sum_3m,uc_amount_sum_6m,uc_amount_mean_3m,uc_amount_mean_6m
0,0_0,2002-09-01 06:21:00,2002-09-01,134.0900,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000
1,0_0,2002-09-01 06:42:00,2002-09-01,38.4800,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000
2,0_0,2002-09-02 06:22:00,2002-09-01,120.3400,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000
3,0_0,2002-09-02 17:45:00,2002-09-01,128.9500,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000
4,0_0,2002-09-03 06:23:00,2002-09-01,104.7100,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000


### **2.3 Variables agregadas por usuario**
---

Cada usuario puede tener una o más tarjetas. Se construyen variables que resumen la relación usuario-tarjeta sin usar directamente el identificador del usuario como predictor.

In [50]:
cards_per_user = (
    df
    .groupby("user")["card"]
    .nunique()
    .reset_index(name="n_cards_user")
)

In [51]:
df = df.merge(cards_per_user, on="user", how="left")

In [53]:
df.head()

,user,card,year,month,day,time,amount,use_chip,merchant_name,merchant_city,merchant_state,zip,mcc,is_fraud,merchant_state_was_missing,zip_was_missing,user_card_id,amount_is_negative,amount_abs,hour,datetime,day_of_week,is_weekend,amount_log,is_online_transaction,is_chip_transaction,is_swipe_transaction,year_month,quarter,day_of_month,uc_tx_count_hist,uc_amount_sum_hist,uc_amount_mean_hist,uc_amount_max_hist,uc_prev_datetime,uc_days_since_prev_tx,uc_tx_count_3m,uc_tx_count_6m,uc_amount_sum_3m,uc_amount_sum_6m,uc_amount_mean_3m,uc_amount_mean_6m,n_cards_user
0,0,0,2002,9,1,06:21,134.0900,Swipe Transaction,3527213246127876953,La Verne,CA,91750,5300,0,0,0,0_0,0,134.0900,6,2002-09-01 06:21:00,6,1,4.9059,0,0,1,2002-09-01,3,1,0,0.0000,0.0000,0.0000,NaT,-1.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,5
1,0,0,2002,9,1,06:42,38.4800,Swipe Transaction,-727612092139916043,Monterey Park,CA,91754,5411,0,0,0,0_0,0,38.4800,6,2002-09-01 06:42:00,6,1,3.6758,0,0,1,2002-09-01,3,1,1,134.0900,134.0900,134.0900,2002-09-01 06:21:00,0.0146,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,5
2,0,0,2002,9,2,06:22,120.3400,Swipe Transaction,-727612092139916043,Monterey Park,CA,91754,5411,0,0,0,0_0,0,120.3400,6,2002-09-02 06:22:00,0,0,4.7986,0,0,1,2002-09-01,3,2,2,172.5700,86.2850,134.0900,2002-09-01 06:42:00,0.9861,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,5
3,0,0,2002,9,2,17:45,128.9500,Swipe Transaction,3414527459579106770,Monterey Park,CA,91754,5651,0,0,0,0_0,0,128.9500,17,2002-09-02 17:45:00,0,0,4.8671,0,0,1,2002-09-01,3,2,3,292.9100,97.6367,134.0900,2002-09-02 06:22:00,0.4743,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,5
4,0,0,2002,9,3,06:23,104.7100,Swipe Transaction,5817218446178736267,La Verne,CA,91750,5912,0,0,0,0_0,0,104.7100,6,2002-09-03 06:23:00,1,0,4.6607,0,0,1,2002-09-01,3,3,4,421.8600,105.4650,134.0900,2002-09-02 17:45:00,0.5264,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,5


### **2.3 Variables de comercio y categoría**
---

Las variables de comercio pueden aportar información relevante, pero algunas tienen alta cardinalidad. Por esta razón, se crean variables de frecuencia en lugar de usar directamente identificadores como `merchant_name`.

In [ ]:
# frecuencia global de transacciones por comerciante
merchant_counts = (
    df["merchant_name"]
    .value_counts(dropna=False)
    .reset_index()
)

merchant_counts.columns = ["merchant_name", "merchant_tx_count_global"]

merchant_counts["merchant_frequency_global"] = (
    merchant_counts["merchant_tx_count_global"] / len(df)
)

In [55]:
df = df.merge(merchant_counts, on="merchant_name", how="left")

In [56]:
# frecuencia global de transacciones por código MCC
mcc_counts = (
    df["mcc"]
    .value_counts(dropna=False)
    .reset_index()
)

mcc_counts.columns = ["mcc", "mcc_tx_count_global"]

mcc_counts["mcc_frequency_global"] = (
    mcc_counts["mcc_tx_count_global"] / len(df)
)

In [57]:
df = df.merge(mcc_counts, on="mcc", how="left")

In [58]:
df[
    [
        "merchant_name",
        "merchant_tx_count_global",
        "merchant_frequency_global",
        "mcc",
        "mcc_tx_count_global",
        "mcc_frequency_global",
    ]
].head()

,merchant_name,merchant_tx_count_global,merchant_frequency_global,mcc,mcc_tx_count_global,mcc_frequency_global
0,3527213246127876953,1574,0.0001,5300,1123037,0.0461
1,-727612092139916043,122367,0.0050,5411,2860738,0.1173
2,-727612092139916043,122367,0.0050,5411,2860738,0.1173
3,3414527459579106770,21018,0.0009,5651,137489,0.0056
4,5817218446178736267,189904,0.0078,5912,1407636,0.0577


In [60]:
df.shape

(24386900, 47)

### **2.3 Variables históricas extendidas por ventanas temporales**
---

In [61]:
df = df.sort_values(["user_card_id", "datetime"]).reset_index(drop=True)
df[["user_card_id", "datetime", "amount_abs", "is_fraud"]].head()

,user_card_id,datetime,amount_abs,is_fraud
0,0_0,2002-09-01 06:21:00,134.0900,0
1,0_0,2002-09-01 06:42:00,38.4800,0
2,0_0,2002-09-02 06:22:00,120.3400,0
3,0_0,2002-09-02 17:45:00,128.9500,0
4,0_0,2002-09-03 06:23:00,104.7100,0


In [62]:
monthly_user_card = (
    df
    .groupby(["user_card_id", "year_month"])
    .agg(
        uc_month_tx_count=("is_fraud", "size"),
        uc_month_amount_sum=("amount_abs", "sum"),
        uc_month_amount_mean=("amount_abs", "mean"),
        uc_month_amount_min=("amount_abs", "min"),
        uc_month_amount_max=("amount_abs", "max"),
        uc_month_amount_std=("amount_abs", "std"),
        uc_month_online_tx_count=("is_online_transaction", "sum"),
        uc_month_negative_tx_count=("amount_is_negative", "sum"),
        uc_month_unique_merchants=("merchant_name", "nunique"),
        uc_month_unique_mcc=("mcc", "nunique"),
    )
    .reset_index()
    .sort_values(["user_card_id", "year_month"])
)

monthly_user_card["uc_month_amount_std"] = monthly_user_card["uc_month_amount_std"].fillna(0)

monthly_user_card.head()

,user_card_id,year_month,uc_month_tx_count,uc_month_amount_sum,uc_month_amount_mean,uc_month_amount_min,uc_month_amount_max,uc_month_amount_std,uc_month_online_tx_count,uc_month_negative_tx_count,uc_month_unique_merchants,uc_month_unique_mcc
0,0_0,2002-09-01,89,"7,691.4400",86.4207,1.1300,199.7100,46.4809,3,2,28,18
1,0_0,2002-10-01,78,"8,183.0900",104.9114,1.8400,"1,049.8200",130.5519,2,6,27,19
2,0_0,2002-11-01,78,"7,833.4400",100.4287,12.0900,379.7300,57.0901,1,3,25,18
3,0_0,2002-12-01,84,"7,669.4900",91.3035,3.6200,255.0000,56.7487,6,4,34,22
4,0_0,2003-01-01,50,"4,341.2300",86.8246,1.3900,432.2200,72.4365,2,1,23,18


In [63]:
def add_rolling_window_features(
    monthly_df: pd.DataFrame,
    group_col: str,
    window_months: list[int],
) -> pd.DataFrame:
    """
    Crea variables históricas por ventanas móviles usando meses completos anteriores.

    Para evitar fuga de información, cada rolling usa shift(1), por lo que
    el mes actual no se incluye en el cálculo de sus propias variables históricas.
    """
    monthly_df = monthly_df.copy()
    monthly_df = monthly_df.sort_values([group_col, "year_month"])

    base_metrics = {
        "uc_month_tx_count": "tx_count",
        "uc_month_amount_sum": "amount_sum",
        "uc_month_amount_mean": "amount_mean",
        "uc_month_amount_min": "amount_min",
        "uc_month_amount_max": "amount_max",
        "uc_month_amount_std": "amount_std",
        "uc_month_online_tx_count": "online_tx_count",
        "uc_month_negative_tx_count": "negative_tx_count",
        "uc_month_unique_merchants": "unique_merchants",
        "uc_month_unique_mcc": "unique_mcc",
    }

    for window in window_months:
        for source_col, output_name in base_metrics.items():
            if source_col in [
                "uc_month_tx_count",
                "uc_month_amount_sum",
                "uc_month_online_tx_count",
                "uc_month_negative_tx_count",
                "uc_month_unique_merchants",
                "uc_month_unique_mcc",
            ]:
                monthly_df[f"uc_{output_name}_{window}m"] = (
                    monthly_df
                    .groupby(group_col)[source_col]
                    .transform(lambda s: s.shift(1).rolling(window=window, min_periods=1).sum())
                )

            elif source_col in [
                "uc_month_amount_mean",
                "uc_month_amount_min",
                "uc_month_amount_max",
                "uc_month_amount_std",
            ]:
                monthly_df[f"uc_{output_name}_{window}m"] = (
                    monthly_df
                    .groupby(group_col)[source_col]
                    .transform(lambda s: s.shift(1).rolling(window=window, min_periods=1).mean())
                )

    return monthly_df

In [64]:
WINDOW_MONTHS = [3, 6, 9, 12]

monthly_user_card = add_rolling_window_features(
    monthly_df=monthly_user_card,
    group_col="user_card_id",
    window_months=WINDOW_MONTHS,
)

monthly_user_card.head()

,user_card_id,year_month,uc_month_tx_count,uc_month_amount_sum,uc_month_amount_mean,uc_month_amount_min,uc_month_amount_max,uc_month_amount_std,uc_month_online_tx_count,uc_month_negative_tx_count,uc_month_unique_merchants,uc_month_unique_mcc,uc_tx_count_3m,uc_amount_sum_3m,uc_amount_mean_3m,uc_amount_min_3m,uc_amount_max_3m,uc_amount_std_3m,uc_online_tx_count_3m,uc_negative_tx_count_3m,uc_unique_merchants_3m,uc_unique_mcc_3m,uc_tx_count_6m,uc_amount_sum_6m,uc_amount_mean_6m,uc_amount_min_6m,uc_amount_max_6m,uc_amount_std_6m,uc_online_tx_count_6m,uc_negative_tx_count_6m,uc_unique_merchants_6m,uc_unique_mcc_6m,uc_tx_count_9m,uc_amount_sum_9m,uc_amount_mean_9m,uc_amount_min_9m,uc_amount_max_9m,uc_amount_std_9m,uc_online_tx_count_9m,uc_negative_tx_count_9m,uc_unique_merchants_9m,uc_unique_mcc_9m,uc_tx_count_12m,uc_amount_sum_12m,uc_amount_mean_12m,uc_amount_min_12m,uc_amount_max_12m,uc_amount_std_12m,uc_online_tx_count_12m,uc_negative_tx_count_12m,uc_unique_merchants_12m,uc_unique_mcc_12m
0,0_0,2002-09-01,89,"7,691.4400",86.4207,1.1300,199.7100,46.4809,3,2,28,18,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,0_0,2002-10-01,78,"8,183.0900",104.9114,1.8400,"1,049.8200",130.5519,2,6,27,19,89.0000,"7,691.4400",86.4207,1.1300,199.7100,46.4809,3.0000,2.0000,28.0000,18.0000,89.0000,"7,691.4400",86.4207,1.1300,199.7100,46.4809,3.0000,2.0000,28.0000,18.0000,89.0000,"7,691.4400",86.4207,1.1300,199.7100,46.4809,3.0000,2.0000,28.0000,18.0000,89.0000,"7,691.4400",86.4207,1.1300,199.7100,46.4809,3.0000,2.0000,28.0000,18.0000
2,0_0,2002-11-01,78,"7,833.4400",100.4287,12.0900,379.7300,57.0901,1,3,25,18,167.0000,"15,874.5300",95.6660,1.4850,624.7650,88.5164,5.0000,8.0000,55.0000,37.0000,167.0000,"15,874.5300",95.6660,1.4850,624.7650,88.5164,5.0000,8.0000,55.0000,37.0000,167.0000,"15,874.5300",95.6660,1.4850,624.7650,88.5164,5.0000,8.0000,55.0000,37.0000,167.0000,"15,874.5300",95.6660,1.4850,624.7650,88.5164,5.0000,8.0000,55.0000,37.0000
3,0_0,2002-12-01,84,"7,669.4900",91.3035,3.6200,255.0000,56.7487,6,4,34,22,245.0000,"23,707.9700",97.2536,5.0200,543.0867,78.0410,6.0000,11.0000,80.0000,55.0000,245.0000,"23,707.9700",97.2536,5.0200,543.0867,78.0410,6.0000,11.0000,80.0000,55.0000,245.0000,"23,707.9700",97.2536,5.0200,543.0867,78.0410,6.0000,11.0000,80.0000,55.0000,245.0000,"23,707.9700",97.2536,5.0200,543.0867,78.0410,6.0000,11.0000,80.0000,55.0000
4,0_0,2003-01-01,50,"4,341.2300",86.8246,1.3900,432.2200,72.4365,2,1,23,18,240.0000,"23,686.0200",98.8812,5.8500,561.5167,81.4636,9.0000,13.0000,86.0000,59.0000,329.0000,"31,377.4600",95.7661,4.6700,471.0650,72.7179,12.0000,15.0000,114.0000,77.0000,329.0000,"31,377.4600",95.7661,4.6700,471.0650,72.7179,12.0000,15.0000,114.0000,77.0000,329.0000,"31,377.4600",95.7661,4.6700,471.0650,72.7179,12.0000,15.0000,114.0000,77.0000


In [65]:
for window in WINDOW_MONTHS:
    tx_col = f"uc_tx_count_{window}m"
    sum_col = f"uc_amount_sum_{window}m"
    mean_col = f"uc_amount_mean_{window}m"
    std_col = f"uc_amount_std_{window}m"
    max_col = f"uc_amount_max_{window}m"
    min_col = f"uc_amount_min_{window}m"
    online_col = f"uc_online_tx_count_{window}m"
    negative_col = f"uc_negative_tx_count_{window}m"

    df_ticket_col = f"uc_avg_ticket_{window}m"
    cv_col = f"uc_amount_cv_{window}m"
    max_mean_ratio_col = f"uc_amount_max_mean_ratio_{window}m"
    min_mean_ratio_col = f"uc_amount_min_mean_ratio_{window}m"
    online_rate_col = f"uc_online_tx_rate_{window}m"
    negative_rate_col = f"uc_negative_tx_rate_{window}m"

    monthly_user_card[df_ticket_col] = (
        monthly_user_card[sum_col] / monthly_user_card[tx_col].replace(0, np.nan)
    )

    monthly_user_card[cv_col] = (
        monthly_user_card[std_col] / monthly_user_card[mean_col].replace(0, np.nan)
    )

    monthly_user_card[max_mean_ratio_col] = (
        monthly_user_card[max_col] / monthly_user_card[mean_col].replace(0, np.nan)
    )

    monthly_user_card[min_mean_ratio_col] = (
        monthly_user_card[min_col] / monthly_user_card[mean_col].replace(0, np.nan)
    )

    monthly_user_card[online_rate_col] = (
        monthly_user_card[online_col] / monthly_user_card[tx_col].replace(0, np.nan)
    )

    monthly_user_card[negative_rate_col] = (
        monthly_user_card[negative_col] / monthly_user_card[tx_col].replace(0, np.nan)
    )

In [66]:
monthly_user_card = monthly_user_card.replace([np.inf, -np.inf], np.nan)
monthly_user_card = monthly_user_card.fillna(0)

### **2.3 Crear métricas de cambio reciente**
---

In [67]:
monthly_user_card["uc_tx_count_ratio_3m_6m"] = (
    monthly_user_card["uc_tx_count_3m"] / monthly_user_card["uc_tx_count_6m"].replace(0, np.nan)
)

monthly_user_card["uc_tx_count_ratio_3m_12m"] = (
    monthly_user_card["uc_tx_count_3m"] / monthly_user_card["uc_tx_count_12m"].replace(0, np.nan)
)

monthly_user_card["uc_tx_count_ratio_6m_12m"] = (
    monthly_user_card["uc_tx_count_6m"] / monthly_user_card["uc_tx_count_12m"].replace(0, np.nan)
)

In [68]:
monthly_user_card["uc_amount_sum_ratio_3m_6m"] = (
    monthly_user_card["uc_amount_sum_3m"] / monthly_user_card["uc_amount_sum_6m"].replace(0, np.nan)
)

monthly_user_card["uc_amount_sum_ratio_3m_12m"] = (
    monthly_user_card["uc_amount_sum_3m"] / monthly_user_card["uc_amount_sum_12m"].replace(0, np.nan)
)

monthly_user_card["uc_amount_sum_ratio_6m_12m"] = (
    monthly_user_card["uc_amount_sum_6m"] / monthly_user_card["uc_amount_sum_12m"].replace(0, np.nan)
)

In [69]:
monthly_user_card["uc_amount_mean_delta_3m_12m"] = (
    monthly_user_card["uc_amount_mean_3m"] - monthly_user_card["uc_amount_mean_12m"]
)

monthly_user_card["uc_amount_mean_ratio_3m_12m"] = (
    monthly_user_card["uc_amount_mean_3m"] / monthly_user_card["uc_amount_mean_12m"].replace(0, np.nan)
)

In [70]:
change_columns = [
    "uc_tx_count_ratio_3m_6m",
    "uc_tx_count_ratio_3m_12m",
    "uc_tx_count_ratio_6m_12m",
    "uc_amount_sum_ratio_3m_6m",
    "uc_amount_sum_ratio_3m_12m",
    "uc_amount_sum_ratio_6m_12m",
    "uc_amount_mean_delta_3m_12m",
    "uc_amount_mean_ratio_3m_12m",
]

monthly_user_card[change_columns] = (
    monthly_user_card[change_columns]
    .replace([np.inf, -np.inf], np.nan)
    .fillna(0)
)

In [71]:
monthly_user_card["uc_no_tx_3m_flag"] = (
    monthly_user_card["uc_tx_count_3m"] == 0
).astype(int)

monthly_user_card["uc_no_tx_6m_flag"] = (
    monthly_user_card["uc_tx_count_6m"] == 0
).astype(int)

monthly_user_card["uc_activity_spike_3m_vs_12m_flag"] = (
    monthly_user_card["uc_tx_count_ratio_3m_12m"] > 0.60
).astype(int)

monthly_user_card["uc_amount_spike_3m_vs_12m_flag"] = (
    monthly_user_card["uc_amount_sum_ratio_3m_12m"] > 0.60
).astype(int)

monthly_user_card["uc_mean_amount_increase_3m_vs_12m_flag"] = (
    monthly_user_card["uc_amount_mean_ratio_3m_12m"] > 1.50
).astype(int)

In [72]:
extended_window_columns = []

for window in WINDOW_MONTHS:
    extended_window_columns.extend([
        f"uc_tx_count_{window}m",
        f"uc_amount_sum_{window}m",
        f"uc_amount_mean_{window}m",
        f"uc_amount_min_{window}m",
        f"uc_amount_max_{window}m",
        f"uc_amount_std_{window}m",
        f"uc_avg_ticket_{window}m",
        f"uc_amount_cv_{window}m",
        f"uc_amount_max_mean_ratio_{window}m",
        f"uc_amount_min_mean_ratio_{window}m",
        f"uc_online_tx_count_{window}m",
        f"uc_online_tx_rate_{window}m",
        f"uc_negative_tx_count_{window}m",
        f"uc_negative_tx_rate_{window}m",
        f"uc_unique_merchants_{window}m",
        f"uc_unique_mcc_{window}m",
    ])

extended_change_columns = [
    "uc_tx_count_ratio_3m_6m",
    "uc_tx_count_ratio_3m_12m",
    "uc_tx_count_ratio_6m_12m",
    "uc_amount_sum_ratio_3m_6m",
    "uc_amount_sum_ratio_3m_12m",
    "uc_amount_sum_ratio_6m_12m",
    "uc_amount_mean_delta_3m_12m",
    "uc_amount_mean_ratio_3m_12m",
    "uc_no_tx_3m_flag",
    "uc_no_tx_6m_flag",
    "uc_activity_spike_3m_vs_12m_flag",
    "uc_amount_spike_3m_vs_12m_flag",
    "uc_mean_amount_increase_3m_vs_12m_flag",
]

monthly_user_card_features_extended = monthly_user_card[
    ["user_card_id", "year_month"]
    + extended_window_columns
    + extended_change_columns
].copy()

monthly_user_card_features_extended.head()

,user_card_id,year_month,uc_tx_count_3m,uc_amount_sum_3m,uc_amount_mean_3m,uc_amount_min_3m,uc_amount_max_3m,uc_amount_std_3m,uc_avg_ticket_3m,uc_amount_cv_3m,uc_amount_max_mean_ratio_3m,uc_amount_min_mean_ratio_3m,uc_online_tx_count_3m,uc_online_tx_rate_3m,uc_negative_tx_count_3m,uc_negative_tx_rate_3m,uc_unique_merchants_3m,uc_unique_mcc_3m,uc_tx_count_6m,uc_amount_sum_6m,uc_amount_mean_6m,uc_amount_min_6m,uc_amount_max_6m,uc_amount_std_6m,uc_avg_ticket_6m,uc_amount_cv_6m,uc_amount_max_mean_ratio_6m,uc_amount_min_mean_ratio_6m,uc_online_tx_count_6m,uc_online_tx_rate_6m,uc_negative_tx_count_6m,uc_negative_tx_rate_6m,uc_unique_merchants_6m,uc_unique_mcc_6m,uc_tx_count_9m,uc_amount_sum_9m,uc_amount_mean_9m,uc_amount_min_9m,uc_amount_max_9m,uc_amount_std_9m,uc_avg_ticket_9m,uc_amount_cv_9m,uc_amount_max_mean_ratio_9m,uc_amount_min_mean_ratio_9m,uc_online_tx_count_9m,uc_online_tx_rate_9m,uc_negative_tx_count_9m,uc_negative_tx_rate_9m,uc_unique_merchants_9m,uc_unique_mcc_9m,uc_tx_count_12m,uc_amount_sum_12m,uc_amount_mean_12m,uc_amount_min_12m,uc_amount_max_12m,uc_amount_std_12m,uc_avg_ticket_12m,uc_amount_cv_12m,uc_amount_max_mean_ratio_12m,uc_amount_min_mean_ratio_12m,uc_online_tx_count_12m,uc_online_tx_rate_12m,uc_negative_tx_count_12m,uc_negative_tx_rate_12m,uc_unique_merchants_12m,uc_unique_mcc_12m,uc_tx_count_ratio_3m_6m,uc_tx_count_ratio_3m_12m,uc_tx_count_ratio_6m_12m,uc_amount_sum_ratio_3m_6m,uc_amount_sum_ratio_3m_12m,uc_amount_sum_ratio_6m_12m,uc_amount_mean_delta_3m_12m,uc_amount_mean_ratio_3m_12m,uc_no_tx_3m_flag,uc_no_tx_6m_flag,uc_activity_spike_3m_vs_12m_flag,uc_amount_spike_3m_vs_12m_flag,uc_mean_amount_increase_3m_vs_12m_flag
0,0_0,2002-09-01,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,1,1,0,0,0
1,0_0,2002-10-01,89.0000,"7,691.4400",86.4207,1.1300,199.7100,46.4809,86.4207,0.5378,2.3109,0.0131,3.0000,0.0337,2.0000,0.0225,28.0000,18.0000,89.0000,"7,691.4400",86.4207,1.1300,199.7100,46.4809,86.4207,0.5378,2.3109,0.0131,3.0000,0.0337,2.0000,0.0225,28.0000,18.0000,89.0000,"7,691.4400",86.4207,1.1300,199.7100,46.4809,86.4207,0.5378,2.3109,0.0131,3.0000,0.0337,2.0000,0.0225,28.0000,18.0000,89.0000,"7,691.4400",86.4207,1.1300,199.7100,46.4809,86.4207,0.5378,2.3109,0.0131,3.0000,0.0337,2.0000,0.0225,28.0000,18.0000,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000,0.0000,1.0000,0,0,1,1,0
2,0_0,2002-11-01,167.0000,"15,874.5300",95.6660,1.4850,624.7650,88.5164,95.0571,0.9253,6.5307,0.0155,5.0000,0.0299,8.0000,0.0479,55.0000,37.0000,167.0000,"15,874.5300",95.6660,1.4850,624.7650,88.5164,95.0571,0.9253,6.5307,0.0155,5.0000,0.0299,8.0000,0.0479,55.0000,37.0000,167.0000,"15,874.5300",95.6660,1.4850,624.7650,88.5164,95.0571,0.9253,6.5307,0.0155,5.0000,0.0299,8.0000,0.0479,55.0000,37.0000,167.0000,"15,874.5300",95.6660,1.4850,624.7650,88.5164,95.0571,0.9253,6.5307,0.0155,5.0000,0.0299,8.0000,0.0479,55.0000,37.0000,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000,0.0000,1.0000,0,0,1,1,0
3,0_0,2002-12-01,245.0000,"23,707.9700",97.2536,5.0200,543.0867,78.0410,96.7672,0.8024,5.5842,0.0516,6.0000,0.0245,11.0000,0.0449,80.0000,55.0000,245.0000,"23,707.9700",97.2536,5.0200,543.0867,78.0410,96.7672,0.8024,5.5842,0.0516,6.0000,0.0245,11.0000,0.0449,80.0000,55.0000,245.0000,"23,707.9700",97.2536,5.0200,543.0867,78.0410,96.7672,0.8024,5.5842,0.0516,6.0000,0.0245,11.0000,0.0449,80.0000,55.0000,245.0000,"23,707.9700",97.2536,5.0200,543.0867,78.0410,96.7672,0.8024,5.5842,0.0516,6.0000,0.0245,11.0000,0.0449,80.0000,55.0000,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000,0.0000,1.0000,0,0,1,1,0
4,0_0,2003-01-01,240

In [73]:
columns_to_drop_before_merge = [
    col for col in monthly_user_card_features_extended.columns
    if col in df.columns and col not in ["user_card_id", "year_month"]
]

columns_to_drop_before_merge

['uc_tx_count_3m',
 'uc_amount_sum_3m',
 'uc_amount_mean_3m',
 'uc_tx_count_6m',
 'uc_amount_sum_6m',
 'uc_amount_mean_6m']

In [74]:
df = df.drop(columns=columns_to_drop_before_merge)

### **2.3 Unión de ventanas al conjunto de datos**
---

In [75]:
df = df.merge(
    monthly_user_card_features_extended,
    on=["user_card_id", "year_month"],
    how="left",
)

In [76]:
df[extended_window_columns + extended_change_columns] = (
    df[extended_window_columns + extended_change_columns]
    .replace([np.inf, -np.inf], np.nan)
    .fillna(0)
)

### **2.3 Variables de flag a nivel de transacción**
---

In [77]:
df["uc_no_history_flag"] = (
    df["uc_tx_count_hist"] == 0
).astype(int)

In [78]:
df["amount_above_hist_max_flag"] = (
    (df["uc_amount_max_hist"] > 0)
    & (df["amount_abs"] > df["uc_amount_max_hist"])
).astype(int)

In [79]:
df["amount_above_6m_max_flag"] = (
    (df["uc_amount_max_6m"] > 0)
    & (df["amount_abs"] > df["uc_amount_max_6m"])
).astype(int)

In [89]:
df["amount_above_3m_max_flag"] = (
    (df["uc_amount_max_3m"] > 0)
    & (df["amount_abs"] > df["uc_amount_max_3m"])
).astype(int)

C:\Users\cecor\AppData\Local\Temp\ipykernel_16684\667150227.py:1: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["amount_above_3m_max_flag"] = (


In [80]:
df["amount_above_6m_max_flag"] = (
    (df["uc_amount_max_6m"] > 0)
    & (df["amount_abs"] > df["uc_amount_max_6m"])
).astype(int)

In [81]:
df["amount_gt_3x_hist_mean_flag"] = (
    (df["uc_amount_mean_hist"] > 0)
    & (df["amount_abs"] > 3 * df["uc_amount_mean_hist"])
).astype(int)

In [82]:
df["amount_gt_3x_12m_mean_flag"] = (
    (df["uc_amount_mean_12m"] > 0)
    & (df["amount_abs"] > 3 * df["uc_amount_mean_12m"])
).astype(int)

### **2.3 Variables de monto vs. histórico**
---

In [83]:
df["amount_to_hist_mean_ratio"] = (
    df["amount_abs"] / df["uc_amount_mean_hist"].replace(0, np.nan)
)

df["amount_to_3m_mean_ratio"] = (
    df["amount_abs"] / df["uc_amount_mean_3m"].replace(0, np.nan)
)

df["amount_to_6m_mean_ratio"] = (
    df["amount_abs"] / df["uc_amount_mean_6m"].replace(0, np.nan)
)

df["amount_to_12m_mean_ratio"] = (
    df["amount_abs"] / df["uc_amount_mean_12m"].replace(0, np.nan)
)

C:\Users\cecor\AppData\Local\Temp\ipykernel_16684\1969745154.py:1: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["amount_to_hist_mean_ratio"] = (
C:\Users\cecor\AppData\Local\Temp\ipykernel_16684\1969745154.py:5: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["amount_to_3m_mean_ratio"] = (
C:\Users\cecor\AppData\Local\Temp\ipykernel_16684\1969745154.py:9: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all col

In [84]:
amount_ratio_columns = [
    "amount_to_hist_mean_ratio",
    "amount_to_3m_mean_ratio",
    "amount_to_6m_mean_ratio",
    "amount_to_12m_mean_ratio",
]

df[amount_ratio_columns] = (
    df[amount_ratio_columns]
    .replace([np.inf, -np.inf], np.nan)
    .fillna(0)
)

### **2.3 Variables flag de inactividad**
---

In [85]:
df["long_inactivity_30d_flag"] = (
    df["uc_days_since_prev_tx"] > 30
).astype(int)

df["long_inactivity_90d_flag"] = (
    df["uc_days_since_prev_tx"] > 90
).astype(int)

df["first_transaction_flag"] = (
    df["uc_days_since_prev_tx"] == -1
).astype(int)

C:\Users\cecor\AppData\Local\Temp\ipykernel_16684\4171814637.py:1: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["long_inactivity_30d_flag"] = (
C:\Users\cecor\AppData\Local\Temp\ipykernel_16684\4171814637.py:5: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["long_inactivity_90d_flag"] = (
C:\Users\cecor\AppData\Local\Temp\ipykernel_16684\4171814637.py:9: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all col

### **2.3 Variables flag de comercios**
---

In [ ]:
df["new_merchant_for_card_flag"] = (
    df.groupby(["user_card_id", "merchant_name"]).cumcount() == 0
).astype(int)

df["new_mcc_for_card_flag"] = (
    df.groupby(["user_card_id", "mcc"]).cumcount() == 0
).astype(int)

### **2.3 Variables transaccional online sin historial transaccional**
---

In [86]:
df["uc_online_tx_count_hist"] = (
    df.groupby("user_card_id")["is_online_transaction"].cumsum()
    - df["is_online_transaction"]
)

C:\Users\cecor\AppData\Local\Temp\ipykernel_16684\153860887.py:1: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["uc_online_tx_count_hist"] = (


In [87]:
df["online_without_online_history_flag"] = (
    (df["is_online_transaction"] == 1)
    & (df["uc_online_tx_count_hist"] == 0)
).astype(int)

C:\Users\cecor\AppData\Local\Temp\ipykernel_16684\339888583.py:1: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["online_without_online_history_flag"] = (


### **2.3 Variables flags de comercio nuevo para la tarjeta**
---

In [91]:
df["new_merchant_for_card_flag"] = (
    df.groupby(["user_card_id", "merchant_name"]).cumcount() == 0
).astype(int)

C:\Users\cecor\AppData\Local\Temp\ipykernel_16684\4255230988.py:1: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["new_merchant_for_card_flag"] = (


In [92]:
df["new_mcc_for_card_flag"] = (
    df.groupby(["user_card_id", "mcc"]).cumcount() == 0
).astype(int)

C:\Users\cecor\AppData\Local\Temp\ipykernel_16684\2885492528.py:1: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["new_mcc_for_card_flag"] = (


In [93]:
new_behavior_columns = (
    extended_window_columns
    + extended_change_columns
    + [
        "uc_no_history_flag",
        "amount_above_hist_max_flag",
        "amount_above_3m_max_flag",
        "amount_above_6m_max_flag",
        "amount_gt_3x_hist_mean_flag",
        "amount_gt_3x_12m_mean_flag",
        "amount_to_hist_mean_ratio",
        "amount_to_3m_mean_ratio",
        "amount_to_6m_mean_ratio",
        "amount_to_12m_mean_ratio",
        "long_inactivity_30d_flag",
        "long_inactivity_90d_flag",
        "first_transaction_flag",
        "new_merchant_for_card_flag",
        "new_mcc_for_card_flag",
        "uc_online_tx_count_hist",
        "online_without_online_history_flag",
    ]
)

df[new_behavior_columns].head()

,uc_tx_count_3m,uc_amount_sum_3m,uc_amount_mean_3m,uc_amount_min_3m,uc_amount_max_3m,uc_amount_std_3m,uc_avg_ticket_3m,uc_amount_cv_3m,uc_amount_max_mean_ratio_3m,uc_amount_min_mean_ratio_3m,uc_online_tx_count_3m,uc_online_tx_rate_3m,uc_negative_tx_count_3m,uc_negative_tx_rate_3m,uc_unique_merchants_3m,uc_unique_mcc_3m,uc_tx_count_6m,uc_amount_sum_6m,uc_amount_mean_6m,uc_amount_min_6m,uc_amount_max_6m,uc_amount_std_6m,uc_avg_ticket_6m,uc_amount_cv_6m,uc_amount_max_mean_ratio_6m,uc_amount_min_mean_ratio_6m,uc_online_tx_count_6m,uc_online_tx_rate_6m,uc_negative_tx_count_6m,uc_negative_tx_rate_6m,uc_unique_merchants_6m,uc_unique_mcc_6m,uc_tx_count_9m,uc_amount_sum_9m,uc_amount_mean_9m,uc_amount_min_9m,uc_amount_max_9m,uc_amount_std_9m,uc_avg_ticket_9m,uc_amount_cv_9m,uc_amount_max_mean_ratio_9m,uc_amount_min_mean_ratio_9m,uc_online_tx_count_9m,uc_online_tx_rate_9m,uc_negative_tx_count_9m,uc_negative_tx_rate_9m,uc_unique_merchants_9m,uc_unique_mcc_9m,uc_tx_count_12m,uc_amount_sum_12m,uc_amount_mean_12m,uc_amount_min_12m,uc_amount_max_12m,uc_amount_std_12m,uc_avg_ticket_12m,uc_amount_cv_12m,uc_amount_max_mean_ratio_12m,uc_amount_min_mean_ratio_12m,uc_online_tx_count_12m,uc_online_tx_rate_12m,uc_negative_tx_count_12m,uc_negative_tx_rate_12m,uc_unique_merchants_12m,uc_unique_mcc_12m,uc_tx_count_ratio_3m_6m,uc_tx_count_ratio_3m_12m,uc_tx_count_ratio_6m_12m,uc_amount_sum_ratio_3m_6m,uc_amount_sum_ratio_3m_12m,uc_amount_sum_ratio_6m_12m,uc_amount_mean_delta_3m_12m,uc_amount_mean_ratio_3m_12m,uc_no_tx_3m_flag,uc_no_tx_6m_flag,uc_activity_spike_3m_vs_12m_flag,uc_amount_spike_3m_vs_12m_flag,uc_mean_amount_increase_3m_vs_12m_flag,uc_no_history_flag,amount_above_hist_max_flag,amount_above_3m_max_flag,amount_above_6m_max_flag,amount_gt_3x_hist_mean_flag,amount_gt_3x_12m_mean_flag,amount_to_hist_mean_ratio,amount_to_3m_mean_ratio,amount_to_6m_mean_ratio,amount_to_12m_mean_ratio,long_inactivity_30d_flag,long_inactivity_90d_flag,first_transaction_flag,new_merchant_for_card_flag,new_mcc_for_card_flag,uc_online_tx_count_hist,online_without_online_history_flag
0,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,1,1,0,0,0,1,0,0,0,0,0,0.0000,0.0000,0.0000,0.0000,0,0,1,1,1,0,0
1,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,1,1,0,0,0,0,0,0,0,0,0,0.2870,0.0000,0.0000,0.0000,0,0,0,1,1,0,0
2,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,1,1,0,0,0,0,0,0,0,0,0,1.3947,0.0000,0.0000,0.0000,0,0,0,0,0,0,0
3,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0

In [94]:
df.shape

(24386900, 135)

### **2.3 Validar nulos e infinitos**
---

In [95]:
new_behavior_nulls = df[new_behavior_columns].isna().sum().sort_values(ascending=False)

new_behavior_nulls.head(20)

uc_tx_count_3m                 0
uc_amount_sum_3m               0
uc_amount_mean_3m              0
uc_amount_min_3m               0
uc_amount_max_3m               0
uc_amount_std_3m               0
uc_avg_ticket_3m               0
uc_amount_cv_3m                0
uc_amount_max_mean_ratio_3m    0
uc_amount_min_mean_ratio_3m    0
uc_online_tx_count_3m          0
uc_online_tx_rate_3m           0
uc_negative_tx_count_3m        0
uc_negative_tx_rate_3m         0
uc_unique_merchants_3m         0
uc_unique_mcc_3m               0
uc_tx_count_6m                 0
uc_amount_sum_6m               0
uc_amount_mean_6m              0
uc_amount_min_6m               0
dtype: int64

In [96]:
assert df[new_behavior_columns].replace([np.inf, -np.inf], np.nan).isna().sum().sum() == 0

### **2.3 Crear columnas limpias auxiliares**
---

In [97]:
df["merchant_state_clean"] = df["merchant_state"].fillna("UNKNOWN").astype(str)
df["merchant_city_clean"] = df["merchant_city"].fillna("UNKNOWN").astype(str)
df["zip_clean"] = df["zip"].fillna("UNKNOWN").astype(str)
df["use_chip_clean"] = df["use_chip"].fillna("UNKNOWN").astype(str)
df["mcc_clean"] = df["mcc"].fillna("UNKNOWN").astype(str)

C:\Users\cecor\AppData\Local\Temp\ipykernel_16684\2343340892.py:1: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["merchant_state_clean"] = df["merchant_state"].fillna("UNKNOWN").astype(str)
C:\Users\cecor\AppData\Local\Temp\ipykernel_16684\2343340892.py:2: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["merchant_city_clean"] = df["merchant_city"].fillna("UNKNOWN").astype(str)
C:\Users\cecor\AppData\Local\Temp\ipykernel_16684\2343340892.py:3: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of 

### **2.3 Flags de horario y monto actual**
---

In [98]:
df["is_night"] = df["hour"].between(0, 5).astype(int)
df["is_morning"] = df["hour"].between(6, 11).astype(int)
df["is_afternoon"] = df["hour"].between(12, 17).astype(int)
df["is_evening"] = df["hour"].between(18, 23).astype(int)

df["is_business_hours"] = df["hour"].between(8, 18).astype(int)
df["is_late_night"] = df["hour"].between(0, 3).astype(int)

C:\Users\cecor\AppData\Local\Temp\ipykernel_16684\2651610818.py:1: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["is_night"] = df["hour"].between(0, 5).astype(int)
C:\Users\cecor\AppData\Local\Temp\ipykernel_16684\2651610818.py:2: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["is_morning"] = df["hour"].between(6, 11).astype(int)
C:\Users\cecor\AppData\Local\Temp\ipykernel_16684\2651610818.py:3: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has po

In [99]:
amount_p90 = df["amount_abs"].quantile(0.90)
amount_p95 = df["amount_abs"].quantile(0.95)
amount_p99 = df["amount_abs"].quantile(0.99)

df["amount_above_global_p90_flag"] = (df["amount_abs"] > amount_p90).astype(int)
df["amount_above_global_p95_flag"] = (df["amount_abs"] > amount_p95).astype(int)
df["amount_above_global_p99_flag"] = (df["amount_abs"] > amount_p99).astype(int)

C:\Users\cecor\AppData\Local\Temp\ipykernel_16684\3700318826.py:5: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["amount_above_global_p90_flag"] = (df["amount_abs"] > amount_p90).astype(int)
C:\Users\cecor\AppData\Local\Temp\ipykernel_16684\3700318826.py:6: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["amount_above_global_p95_flag"] = (df["amount_abs"] > amount_p95).astype(int)
C:\Users\cecor\AppData\Local\Temp\ipykernel_16684\3700318826.py:7: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result

In [100]:
df["amount_is_zero_flag"] = (df["amount_abs"] == 0).astype(int)
df["amount_below_1_flag"] = (df["amount_abs"] < 1).astype(int)
df["amount_below_5_flag"] = (df["amount_abs"] < 5).astype(int)

df["amount_is_round_10_flag"] = np.isclose(df["amount_abs"] % 10, 0, atol=1e-6).astype(int)
df["amount_is_round_100_flag"] = np.isclose(df["amount_abs"] % 100, 0, atol=1e-6).astype(int)

C:\Users\cecor\AppData\Local\Temp\ipykernel_16684\3170149005.py:1: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["amount_is_zero_flag"] = (df["amount_abs"] == 0).astype(int)
C:\Users\cecor\AppData\Local\Temp\ipykernel_16684\3170149005.py:2: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["amount_below_1_flag"] = (df["amount_abs"] < 1).astype(int)
C:\Users\cecor\AppData\Local\Temp\ipykernel_16684\3170149005.py:3: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many tim

### **2.3 Ratios de monto frente a ventanas**
---

In [101]:
for window in [3, 6, 9, 12]:
    mean_col = f"uc_amount_mean_{window}m"
    max_col = f"uc_amount_max_{window}m"
    min_col = f"uc_amount_min_{window}m"
    std_col = f"uc_amount_std_{window}m"

    df[f"amount_to_{window}m_mean_ratio"] = (
        df["amount_abs"] / df[mean_col].replace(0, np.nan)
    )

    df[f"amount_to_{window}m_max_ratio"] = (
        df["amount_abs"] / df[max_col].replace(0, np.nan)
    )

    df[f"amount_minus_{window}m_mean"] = (
        df["amount_abs"] - df[mean_col]
    )

    df[f"amount_zscore_{window}m"] = (
        (df["amount_abs"] - df[mean_col]) / df[std_col].replace(0, np.nan)
    )

    df[f"amount_above_{window}m_mean_flag"] = (
        (df[mean_col] > 0) & (df["amount_abs"] > df[mean_col])
    ).astype(int)

    df[f"amount_above_2x_{window}m_mean_flag"] = (
        (df[mean_col] > 0) & (df["amount_abs"] > 2 * df[mean_col])
    ).astype(int)

    df[f"amount_above_3x_{window}m_mean_flag"] = (
        (df[mean_col] > 0) & (df["amount_abs"] > 3 * df[mean_col])
    ).astype(int)

    df[f"amount_above_{window}m_max_flag"] = (
        (df[max_col] > 0) & (df["amount_abs"] > df[max_col])
    ).astype(int)

C:\Users\cecor\AppData\Local\Temp\ipykernel_16684\1342233481.py:11: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[f"amount_to_{window}m_max_ratio"] = (
C:\Users\cecor\AppData\Local\Temp\ipykernel_16684\1342233481.py:15: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[f"amount_minus_{window}m_mean"] = (
C:\Users\cecor\AppData\Local\Temp\ipykernel_16684\1342233481.py:19: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider jo

In [102]:
amount_window_comparison_cols = [
    col for col in df.columns
    if (
        col.startswith("amount_to_")
        or col.startswith("amount_minus_")
        or col.startswith("amount_zscore_")
        or col.startswith("amount_above_")
    )
]

df[amount_window_comparison_cols] = (
    df[amount_window_comparison_cols]
    .replace([np.inf, -np.inf], np.nan)
    .fillna(0)
)

### **2.3 Historial de canal por user-card**
---

In [103]:
df["uc_online_tx_count_hist"] = (
    df.groupby("user_card_id")["is_online_transaction"].cumsum()
    - df["is_online_transaction"]
)

df["uc_chip_tx_count_hist"] = (
    df.groupby("user_card_id")["is_chip_transaction"].cumsum()
    - df["is_chip_transaction"]
)

df["uc_swipe_tx_count_hist"] = (
    df.groupby("user_card_id")["is_swipe_transaction"].cumsum()
    - df["is_swipe_transaction"]
)

C:\Users\cecor\AppData\Local\Temp\ipykernel_16684\279514362.py:6: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["uc_chip_tx_count_hist"] = (
C:\Users\cecor\AppData\Local\Temp\ipykernel_16684\279514362.py:11: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["uc_swipe_tx_count_hist"] = (


In [104]:
df["uc_online_tx_rate_hist"] = (
    df["uc_online_tx_count_hist"] / df["uc_tx_count_hist"].replace(0, np.nan)
)

df["uc_chip_tx_rate_hist"] = (
    df["uc_chip_tx_count_hist"] / df["uc_tx_count_hist"].replace(0, np.nan)
)

df["uc_swipe_tx_rate_hist"] = (
    df["uc_swipe_tx_count_hist"] / df["uc_tx_count_hist"].replace(0, np.nan)
)

C:\Users\cecor\AppData\Local\Temp\ipykernel_16684\3400672656.py:1: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["uc_online_tx_rate_hist"] = (
C:\Users\cecor\AppData\Local\Temp\ipykernel_16684\3400672656.py:5: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["uc_chip_tx_rate_hist"] = (
C:\Users\cecor\AppData\Local\Temp\ipykernel_16684\3400672656.py:9: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns a

In [105]:
channel_hist_cols = [
    "uc_online_tx_count_hist",
    "uc_chip_tx_count_hist",
    "uc_swipe_tx_count_hist",
    "uc_online_tx_rate_hist",
    "uc_chip_tx_rate_hist",
    "uc_swipe_tx_rate_hist",
]

df[channel_hist_cols] = (
    df[channel_hist_cols]
    .replace([np.inf, -np.inf], np.nan)
    .fillna(0)
)

In [106]:
df["online_first_time_for_card_flag"] = (
    (df["is_online_transaction"] == 1)
    & (df["uc_online_tx_count_hist"] == 0)
).astype(int)

df["chip_first_time_for_card_flag"] = (
    (df["is_chip_transaction"] == 1)
    & (df["uc_chip_tx_count_hist"] == 0)
).astype(int)

df["swipe_first_time_for_card_flag"] = (
    (df["is_swipe_transaction"] == 1)
    & (df["uc_swipe_tx_count_hist"] == 0)
).astype(int)

C:\Users\cecor\AppData\Local\Temp\ipykernel_16684\1644813602.py:1: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["online_first_time_for_card_flag"] = (
C:\Users\cecor\AppData\Local\Temp\ipykernel_16684\1644813602.py:6: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["chip_first_time_for_card_flag"] = (
C:\Users\cecor\AppData\Local\Temp\ipykernel_16684\1644813602.py:11: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider jo

### **2.3 Variables de novedad por comercio, MCC, ciudad, estado y ZIP**
---

In [107]:
df["new_merchant_for_card_flag"] = (
    df.groupby(["user_card_id", "merchant_name"]).cumcount() == 0
).astype(int)

df["new_mcc_for_card_flag"] = (
    df.groupby(["user_card_id", "mcc_clean"]).cumcount() == 0
).astype(int)

df["new_city_for_card_flag"] = (
    df.groupby(["user_card_id", "merchant_city_clean"]).cumcount() == 0
).astype(int)

df["new_state_for_card_flag"] = (
    df.groupby(["user_card_id", "merchant_state_clean"]).cumcount() == 0
).astype(int)

df["new_zip_for_card_flag"] = (
    df.groupby(["user_card_id", "zip_clean"]).cumcount() == 0
).astype(int)

C:\Users\cecor\AppData\Local\Temp\ipykernel_16684\4276820426.py:9: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["new_city_for_card_flag"] = (
C:\Users\cecor\AppData\Local\Temp\ipykernel_16684\4276820426.py:13: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["new_state_for_card_flag"] = (
C:\Users\cecor\AppData\Local\Temp\ipykernel_16684\4276820426.py:17: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all colu

In [108]:
df["uc_unique_merchants_hist"] = (
    df.groupby("user_card_id")["new_merchant_for_card_flag"].cumsum()
    - df["new_merchant_for_card_flag"]
)

df["uc_unique_mcc_hist"] = (
    df.groupby("user_card_id")["new_mcc_for_card_flag"].cumsum()
    - df["new_mcc_for_card_flag"]
)

df["uc_unique_cities_hist"] = (
    df.groupby("user_card_id")["new_city_for_card_flag"].cumsum()
    - df["new_city_for_card_flag"]
)

df["uc_unique_states_hist"] = (
    df.groupby("user_card_id")["new_state_for_card_flag"].cumsum()
    - df["new_state_for_card_flag"]
)

df["uc_unique_zips_hist"] = (
    df.groupby("user_card_id")["new_zip_for_card_flag"].cumsum()
    - df["new_zip_for_card_flag"]
)

C:\Users\cecor\AppData\Local\Temp\ipykernel_16684\2836784457.py:1: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["uc_unique_merchants_hist"] = (
C:\Users\cecor\AppData\Local\Temp\ipykernel_16684\2836784457.py:6: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["uc_unique_mcc_hist"] = (
C:\Users\cecor\AppData\Local\Temp\ipykernel_16684\2836784457.py:11: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns 

In [109]:
df["uc_merchant_tx_count_hist"] = (
    df.groupby(["user_card_id", "merchant_name"]).cumcount()
)

df["uc_mcc_tx_count_hist"] = (
    df.groupby(["user_card_id", "mcc_clean"]).cumcount()
)

df["uc_city_tx_count_hist"] = (
    df.groupby(["user_card_id", "merchant_city_clean"]).cumcount()
)

df["uc_state_tx_count_hist"] = (
    df.groupby(["user_card_id", "merchant_state_clean"]).cumcount()
)

df["uc_zip_tx_count_hist"] = (
    df.groupby(["user_card_id", "zip_clean"]).cumcount()
)

C:\Users\cecor\AppData\Local\Temp\ipykernel_16684\2616559204.py:1: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["uc_merchant_tx_count_hist"] = (
C:\Users\cecor\AppData\Local\Temp\ipykernel_16684\2616559204.py:5: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["uc_mcc_tx_count_hist"] = (
C:\Users\cecor\AppData\Local\Temp\ipykernel_16684\2616559204.py:9: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all column

In [110]:
df["first_time_merchant_card_flag"] = (df["uc_merchant_tx_count_hist"] == 0).astype(int)
df["first_time_mcc_card_flag"] = (df["uc_mcc_tx_count_hist"] == 0).astype(int)
df["first_time_city_card_flag"] = (df["uc_city_tx_count_hist"] == 0).astype(int)
df["first_time_state_card_flag"] = (df["uc_state_tx_count_hist"] == 0).astype(int)
df["first_time_zip_card_flag"] = (df["uc_zip_tx_count_hist"] == 0).astype(int)

C:\Users\cecor\AppData\Local\Temp\ipykernel_16684\924968635.py:1: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["first_time_merchant_card_flag"] = (df["uc_merchant_tx_count_hist"] == 0).astype(int)
C:\Users\cecor\AppData\Local\Temp\ipykernel_16684\924968635.py:2: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["first_time_mcc_card_flag"] = (df["uc_mcc_tx_count_hist"] == 0).astype(int)
C:\Users\cecor\AppData\Local\Temp\ipykernel_16684\924968635.py:3: PerformanceWarning: DataFrame is highly fragmented.  This is usually the res

### **2.3 Variables históricas por usuario**
---

In [111]:
df = df.sort_values(["user", "datetime"]).reset_index(drop=True)

In [112]:
df["user_tx_count_hist"] = df.groupby("user").cumcount()

df["user_amount_sum_hist"] = (
    df.groupby("user")["amount_abs"].cumsum() - df["amount_abs"]
)

df["user_amount_mean_hist"] = (
    df["user_amount_sum_hist"] / df["user_tx_count_hist"].replace(0, np.nan)
)

df["user_amount_mean_hist"] = df["user_amount_mean_hist"].fillna(0)

C:\Users\cecor\AppData\Local\Temp\ipykernel_16684\3082016061.py:1: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["user_tx_count_hist"] = df.groupby("user").cumcount()
C:\Users\cecor\AppData\Local\Temp\ipykernel_16684\3082016061.py:3: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["user_amount_sum_hist"] = (
C:\Users\cecor\AppData\Local\Temp\ipykernel_16684\3082016061.py:7: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consid

In [113]:
df["user_amount_max_hist"] = (
    df.groupby("user")["amount_abs"]
    .cummax()
    .groupby(df["user"])
    .shift(1)
    .fillna(0)
)

C:\Users\cecor\AppData\Local\Temp\ipykernel_16684\1177486784.py:1: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["user_amount_max_hist"] = (


In [114]:
df["user_prev_datetime"] = df.groupby("user")["datetime"].shift(1)

df["user_days_since_prev_tx"] = (
    (df["datetime"] - df["user_prev_datetime"])
    .dt.total_seconds()
    .div(86400)
)

df["user_days_since_prev_tx"] = df["user_days_since_prev_tx"].fillna(-1)

C:\Users\cecor\AppData\Local\Temp\ipykernel_16684\13638241.py:1: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["user_prev_datetime"] = df.groupby("user")["datetime"].shift(1)
C:\Users\cecor\AppData\Local\Temp\ipykernel_16684\13638241.py:3: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["user_days_since_prev_tx"] = (


In [115]:
df["amount_to_user_hist_mean_ratio"] = (
    df["amount_abs"] / df["user_amount_mean_hist"].replace(0, np.nan)
)

df["amount_to_user_hist_max_ratio"] = (
    df["amount_abs"] / df["user_amount_max_hist"].replace(0, np.nan)
)

df["amount_above_user_hist_max_flag"] = (
    (df["user_amount_max_hist"] > 0)
    & (df["amount_abs"] > df["user_amount_max_hist"])
).astype(int)

C:\Users\cecor\AppData\Local\Temp\ipykernel_16684\259133665.py:1: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["amount_to_user_hist_mean_ratio"] = (
C:\Users\cecor\AppData\Local\Temp\ipykernel_16684\259133665.py:5: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["amount_to_user_hist_max_ratio"] = (
C:\Users\cecor\AppData\Local\Temp\ipykernel_16684\259133665.py:9: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining

In [116]:
user_hist_cols = [
    "user_tx_count_hist",
    "user_amount_sum_hist",
    "user_amount_mean_hist",
    "user_amount_max_hist",
    "user_days_since_prev_tx",
    "amount_to_user_hist_mean_ratio",
    "amount_to_user_hist_max_ratio",
    "amount_above_user_hist_max_flag",
]

df[user_hist_cols] = (
    df[user_hist_cols]
    .replace([np.inf, -np.inf], np.nan)
    .fillna(0)
)

In [ ]:
df = df.sort_values(["user_card_id", "datetime"]).reset_index(drop=True)

### **2.3 Variables de velocidad intradía**
---

In [ ]:
df["transaction_date"] = df["datetime"].dt.date
df["uc_same_day_tx_count_prev"] = (
    df.groupby(["user_card_id", "transaction_date"]).cumcount()
)

In [ ]:
df["uc_same_day_amount_sum_prev"] = (
    df.groupby(["user_card_id", "transaction_date"])["amount_abs"].cumsum()
    - df["amount_abs"]
)

In [ ]:
df["uc_same_day_amount_mean_prev"] = (
    df["uc_same_day_amount_sum_prev"] / df["uc_same_day_tx_count_prev"].replace(0, np.nan)
)

df["uc_same_day_amount_mean_prev"] = df["uc_same_day_amount_mean_prev"].fillna(0)

In [ ]:
df["multiple_tx_same_day_flag"] = (
    df["uc_same_day_tx_count_prev"] >= 1
).astype(int)

df["high_velocity_same_day_flag"] = (
    df["uc_same_day_tx_count_prev"] >= 5
).astype(int)

df["very_high_velocity_same_day_flag"] = (
    df["uc_same_day_tx_count_prev"] >= 10
).astype(int)

### **2.3 Frecuencias globales adicionales**
---

In [ ]:
state_counts = df["merchant_state_clean"].value_counts(dropna=False).reset_index()
state_counts.columns = ["merchant_state_clean", "merchant_state_tx_count_global"]
state_counts["merchant_state_frequency_global"] = (
    state_counts["merchant_state_tx_count_global"] / len(df)
)

df = df.merge(state_counts, on="merchant_state_clean", how="left")

In [ ]:
city_counts = df["merchant_city_clean"].value_counts(dropna=False).reset_index()
city_counts.columns = ["merchant_city_clean", "merchant_city_tx_count_global"]
city_counts["merchant_city_frequency_global"] = (
    city_counts["merchant_city_tx_count_global"] / len(df)
)

df = df.merge(city_counts, on="merchant_city_clean", how="left")

In [ ]:
zip_counts = df["zip_clean"].value_counts(dropna=False).reset_index()
zip_counts.columns = ["zip_clean", "zip_tx_count_global"]
zip_counts["zip_frequency_global"] = (
    zip_counts["zip_tx_count_global"] / len(df)
)

df = df.merge(zip_counts, on="zip_clean", how="left")

In [ ]:
use_chip_counts = df["use_chip_clean"].value_counts(dropna=False).reset_index()
use_chip_counts.columns = ["use_chip_clean", "use_chip_tx_count_global"]
use_chip_counts["use_chip_frequency_global"] = (
    use_chip_counts["use_chip_tx_count_global"] / len(df)
)

df = df.merge(use_chip_counts, on="use_chip_clean", how="left")

In [ ]:
use_chip_counts = df["use_chip_clean"].value_counts(dropna=False).reset_index()
use_chip_counts.columns = ["use_chip_clean", "use_chip_tx_count_global"]
use_chip_counts["use_chip_frequency_global"] = (
    use_chip_counts["use_chip_tx_count_global"] / len(df)
)

df = df.merge(use_chip_counts, on="use_chip_clean", how="left")

### **2.3 Pasos finales**
---

In [ ]:
additional_behavior_columns = [
    "is_night",
    "is_morning",
    "is_afternoon",
    "is_evening",
    "is_business_hours",
    "is_late_night",
    "amount_above_global_p90_flag",
    "amount_above_global_p95_flag",
    "amount_above_global_p99_flag",
    "amount_is_zero_flag",
    "amount_below_1_flag",
    "amount_below_5_flag",
    "amount_is_round_10_flag",
    "amount_is_round_100_flag",

    "uc_online_tx_count_hist",
    "uc_chip_tx_count_hist",
    "uc_swipe_tx_count_hist",
    "uc_online_tx_rate_hist",
    "uc_chip_tx_rate_hist",
    "uc_swipe_tx_rate_hist",
    "online_first_time_for_card_flag",
    "chip_first_time_for_card_flag",
    "swipe_first_time_for_card_flag",

    "new_merchant_for_card_flag",
    "new_mcc_for_card_flag",
    "new_city_for_card_flag",
    "new_state_for_card_flag",
    "new_zip_for_card_flag",
    "uc_unique_merchants_hist",
    "uc_unique_mcc_hist",
    "uc_unique_cities_hist",
    "uc_unique_states_hist",
    "uc_unique_zips_hist",

    "uc_merchant_tx_count_hist",
    "uc_mcc_tx_count_hist",
    "uc_city_tx_count_hist",
    "uc_state_tx_count_hist",
    "uc_zip_tx_count_hist",
    "first_time_merchant_card_flag",
    "first_time_mcc_card_flag",
    "first_time_city_card_flag",
    "first_time_state_card_flag",
    "first_time_zip_card_flag",

    "user_tx_count_hist",
    "user_amount_sum_hist",
    "user_amount_mean_hist",
    "user_amount_max_hist",
    "user_days_since_prev_tx",
    "amount_to_user_hist_mean_ratio",
    "amount_to_user_hist_max_ratio",
    "amount_above_user_hist_max_flag",

    "uc_same_day_tx_count_prev",
    "uc_same_day_amount_sum_prev",
    "uc_same_day_amount_mean_prev",
    "multiple_tx_same_day_flag",
    "high_velocity_same_day_flag",
    "very_high_velocity_same_day_flag",

    "merchant_state_tx_count_global",
    "merchant_state_frequency_global",
    "merchant_city_tx_count_global",
    "merchant_city_frequency_global",
    "zip_tx_count_global",
    "zip_frequency_global",
    "use_chip_tx_count_global",
    "use_chip_frequency_global",
    "rare_merchant_flag",
    "rare_mcc_flag",
    "rare_city_flag",
    "rare_state_flag",
    "rare_zip_flag",
]

In [ ]:
additional_behavior_columns = (
    additional_behavior_columns
    + amount_window_comparison_cols
)

In [ ]:
additional_behavior_columns = list(dict.fromkeys(additional_behavior_columns))

len(additional_behavior_columns)

### **2.3 Validar nulos e infinitos**
---

In [ ]:
df[additional_behavior_columns] = (
    df[additional_behavior_columns]
    .replace([np.inf, -np.inf], np.nan)
    .fillna(0)
)

In [ ]:
additional_nulls = df[additional_behavior_columns].isna().sum().sort_values(ascending=False)

additional_nulls.head(20)

In [ ]:
assert df[additional_behavior_columns].replace([np.inf, -np.inf], np.nan).isna().sum().sum() == 0

In [ ]:
current_feature_candidates = (
    new_behavior_columns
    + additional_behavior_columns
)

current_feature_candidates = list(dict.fromkeys(current_feature_candidates))

len(current_feature_candidates)

### **2.3 Pasos finales**
---

In [ ]:
numeric_features = [
    "amount_abs",
    "amount_log",
    "amount_is_negative",
    "merchant_state_was_missing",
    "zip_was_missing",
    "hour",
    "day_of_week",
    "is_weekend",
    "quarter",
    "day_of_month",
    "n_cards_user",

    "uc_tx_count_hist",
    "uc_amount_sum_hist",
    "uc_amount_mean_hist",
    "uc_amount_max_hist",
    "uc_days_since_prev_tx",

    "merchant_tx_count_global",
    "merchant_frequency_global",
    "mcc_tx_count_global",
    "mcc_frequency_global",
] + new_behavior_columns + additional_behavior_columns

In [ ]:
numeric_features = list(dict.fromkeys(numeric_features))

len(numeric_features)

### **2.3 Pasos finales**
---

### **2.3 Pasos finales**
---

### **2.3 Pasos finales**
---

### **2.3 Pasos finales**
---

### **2.3 Pasos finales**
---

### **2.3 Pasos finales**
---

### **2.3 Pasos finales**
---